In [ ]:
!pip install gensim

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline

from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize


from wordcloud import WordCloud

from gensim import matutils, models
import scipy.sparse

import re
import string

import nltk
from nltk import pos_tag
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import WordNetLemmatizer


In [ ]:
# Let's read in our document-term matrix
speech_df = pd.read_csv('lemma.csv')
data = pd.read_csv('data_dtm_lemma.csv')
tdm = data.transpose()
tdm.shape

(33479, 278)

In [ ]:
tdm = tdm.iloc[1: , :]
tdm.head(10)

,0,1,2,3,4,5,6,7,8,9,...,268,269,270,271,272,273,274,275,276,277
aa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aahhhh,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aaron,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aback,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
abalthus,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
abandon,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
abate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
abbreviations,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
abc,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
abcsbut,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
speech_df = speech_df.iloc[: , 1:]
speech_df.head()

,Unnamed: 0,speaker,year,transcript,length
0,0,OPRAH WINFREY,1918.0,thank wallis annenberg special thank dean will...,15301.0
1,1,OPRAH WINFREY,2007.0,president swygert trustees distinguish guests ...,14168.0
2,2,FRANKLIN D ROOSEVELT,1932.0,day honorable attainment honor confer upon dee...,16017.0
3,3,WILLIAM ALLEN,1936.0,commencement orator auditors turn face around ...,14953.0
4,4,CARRIE CHAPMAN,1936.0,bring message sweet briar college especially s...,22942.0


In [ ]:
speech_df = speech_df.iloc[: , 1:]
speech_df.head()

,speaker,year,transcript,length
0,OPRAH WINFREY,1918.0,thank wallis annenberg special thank dean will...,15301.0
1,OPRAH WINFREY,2007.0,president swygert trustees distinguish guests ...,14168.0
2,FRANKLIN D ROOSEVELT,1932.0,day honorable attainment honor confer upon dee...,16017.0
3,WILLIAM ALLEN,1936.0,commencement orator auditors turn face around ...,14953.0
4,CARRIE CHAPMAN,1936.0,bring message sweet briar college especially s...,22942.0


In [ ]:
# Bag of words with CountVectorizer
# add_stop_words selected from after lemmatization
# will also remove common_words (most commonly used words in all speeches)
# will also remove boring words (words that do not add much insight to topic modeling)

add_stop_words = ['like','youre','ive','im','really','id','ve','just','dont','thi','wa',
                  'say','know','make','people']

boring_words = ['say','like','just','dont','don','im',
                  'ive','youll','youve','things','thing','youre','right','really','lot',
                  'make','know','people','way','day','class']

common_words = ['make','know','say','time','people','life','think','like','world','years','want',
                'come','work','dont','live','tell','today','im','way','youre', 'day','new','things',
                'right', 'graduate','good','learn','great','look']


add_stop_words = add_stop_words + boring_words + common_words

stop_words = text.ENGLISH_STOP_WORDS.union(add_stop_words)

cv = CountVectorizer(stop_words=list(stop_words))
data_cv = cv.fit_transform(speech_df.transcript)

In [ ]:
# If add_stop_words is modified, update tdm
data_dtm = pd.DataFrame(data_cv.toarray(), columns=cv.get_feature_names_out())
data_dtm.index = speech_df.index
data_dtm = data_dtm.iloc[:,:-1]
tdm = data_dtm.transpose()

In [ ]:
# We're going to put the term-document matrix into a new gensim format
# From df --> sparse matrix --> gensim corpus
sparse_counts = scipy.sparse.csr_matrix(tdm)
corpus = matutils.Sparse2Corpus(sparse_counts)

In [ ]:
# Gensim also requires dictionary of the all terms and their respective location in the term-document matrix
# {dictionsry of location: word}
id2word = dict((v, k) for k, v in cv.vocabulary_.items())
len(id2word)

33441

#LDA

##All text

In [ ]:
def get_lda_topics(model, num_topics):
    """Print lda topics with pd.DataFrame"""

    word_dict = {}
    for i in range(num_topics):
        words = model.show_topic(i, topn = 10)
        word_dict['Topic #' + '{:02d}'.format(i+1)] = [i[0] for i in words]

    return pd.DataFrame(word_dict).transpose()

In [ ]:
# We need to specify two parameters: the number of topics and the number of passes
num_topics = 2
lda = models.LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, passes=10)
get_lda_topics(lda, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,love,school,need,mean,college,thats,write,ask,try,help
Topic #02,change,thats,ask,school,little,job,mean,need,start,use


In [ ]:
# LDA for num_topics = 3
num_topics = 3
lda = models.LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, passes=10)
get_lda_topics(lda, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,need,school,change,thats,mean,help,try,human,state,ask
Topic #02,school,women,mean,need,love,write,college,ask,job,little
Topic #03,love,school,college,ask,need,start,thats,talk,job,didnt


In [ ]:
# LDA for num_topics = 4
num_topics = 4
lda = models.LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, passes=10)
get_lda_topics(lda, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,school,university,college,mean,education,need,women,help,let,thats
Topic #02,love,thats,try,mean,need,little,talk,ask,start,let
Topic #03,need,love,change,mean,write,dream,thank,little,help,ask
Topic #04,school,job,ask,remember,thats,love,need,write,didnt,start


##Nouns Only

In [ ]:
# Let's create a function to pull out nouns from a string of text
from nltk import word_tokenize, pos_tag

def nouns(text):
    '''Given a string of text, tokenize the text and pull out only the nouns.'''
    is_noun = lambda pos: pos[:2] == 'NN'
    tokenized = word_tokenize(text)
    all_nouns = [word for (word, pos) in pos_tag(tokenized) if is_noun(pos)]
    return ' '.join(all_nouns)

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
# Apply the nouns function to the transcripts to filter only on nouns
speech_df['nouns'] = speech_df.transcript.apply(nouns)
speech_df.head()

,speaker,year,transcript,length,nouns
0,OPRAH WINFREY,1918.0,thank wallis annenberg special thank dean will...,15301.0,thank wallis dean bay invite today parent facu...
1,OPRAH WINFREY,2007.0,president swygert trustees distinguish guests ...,14168.0,president swygert trustees guests honorees dr ...
2,FRANKLIN D ROOSEVELT,1932.0,day honorable attainment honor confer upon dee...,16017.0,day attainment honor share satisfaction laurel...
3,WILLIAM ALLEN,1936.0,commencement orator auditors turn face around ...,14953.0,commencement orator auditors look world think ...
4,CARRIE CHAPMAN,1936.0,bring message sweet briar college especially s...,22942.0,message college class portals world message br...


In [ ]:
# Create dtm_n (document-term matrix with nouns only)
cv_n = CountVectorizer(stop_words=list(stop_words))
data_cv_n = cv_n.fit_transform(speech_df.nouns)

dtm_n = pd.DataFrame(data_cv_n.toarray(), columns=cv_n.get_feature_names_out())
dtm_n.index = speech_df.index
dtm_n = dtm_n.iloc[:,:-1]
# dtm_n

In [ ]:
# Create the gensim corpus
corpus_n = matutils.Sparse2Corpus(scipy.sparse.csr_matrix(dtm_n.transpose()))

# Create the vocabulary dictionary
id2word_n = dict((v, k) for k, v in cv_n.vocabulary_.items())

In [ ]:
# Let's start with 2 topics
num_topics = 2
lda_n = models.LdaModel(corpus=corpus_n, num_topics=num_topics, id2word=id2word_n, passes=10)
get_lda_topics(lda_n, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,school,thats,college,education,country,university,state,job,place,parent
Topic #02,school,job,women,college,thats,year,dream,parent,family,change


In [ ]:
# 3 topics
num_topics = 3
lda_n = models.LdaModel(corpus=corpus_n, num_topics=num_topics, id2word=id2word_n, passes=10)
get_lda_topics(lda_n, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,school,thats,college,job,parent,talk,university,education,dream,course
Topic #02,job,school,thats,person,family,parent,college,man,didnt,change
Topic #03,women,school,college,job,state,place,change,education,man,country


In [ ]:
# 4 topics
num_topics = 4
lda_n = models.LdaModel(corpus=corpus_n, num_topics=num_topics, id2word=id2word_n, passes=10)
get_lda_topics(lda_n, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,college,thats,school,dream,parent,change,job,person,talk,family
Topic #02,school,college,job,thats,year,women,parent,word,talk,didnt
Topic #03,job,school,thats,women,college,talk,year,place,didnt,success
Topic #04,school,education,state,country,university,war,man,college,place,government


In [ ]:
# 10 topics
num_topics = 10
lda_n = models.LdaModel(corpus=corpus_n, num_topics=num_topics, id2word=id2word_n, passes=30)
get_lda_topics(lda_n, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,school,word,job,gift,college,talk,thats,year,parent,music
Topic #02,job,school,thats,college,parent,didnt,career,year,company,talk
Topic #03,women,school,place,college,thats,year,state,talk,man,parent
Topic #04,women,men,success,school,college,peace,workers,fact,nations,advice
Topic #05,question,word,course,job,water,chance,thats,house,search,information
Topic #06,dream,university,thats,god,school,family,thank,place,education,college
Topic #07,school,thats,course,family,job,college,parent,change,year,didnt
Topic #08,war,education,country,state,man,generation,college,place,history,course
Topic #09,advice,thats,parent,eye,job,identity,talk,school,law,friends
Topic #10,college,school,thats,change,state,education,government,women,university,help


In [ ]:
# 20 topics
num_topics = 20
lda_n = models.LdaModel(corpus=corpus_n, num_topics=num_topics, id2word=id2word_n, passes=40)
get_lda_topics(lda_n, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,parent,college,advice,generation,job,thats,word,talk,president,school
Topic #02,job,school,company,business,thats,career,university,passion,year,kind
Topic #03,dream,school,thats,college,job,course,question,moment,journey,technology
Topic #04,dad,legacy,dream,matter,thats,man,room,help,dartmouth,talk
Topic #05,university,college,president,person,state,moment,thats,deaf,change,school
Topic #06,college,place,reason,business,experience,water,mit,school,fear,question
Topic #07,man,state,society,education,freedom,country,power,progress,energy,countries
Topic #08,school,dream,family,gift,talk,college,use,help,change,parent
Topic #09,job,thats,parent,didnt,guy,school,team,family,dream,year
Topic #10,women,thats,men,talk,woman,college,school,change,man,word


##Nouns and Adjectives

In [ ]:
# Let's create a function to pull out nouns from a string of text
from nltk import word_tokenize, pos_tag

def nouns_adj(text):
    '''Given a string of text, tokenize the text and pull out only the nouns.'''
    is_noun = lambda pos: pos[:2] == 'NN' or pos[:2] == 'JJ'
    tokenized = word_tokenize(text)
    all_nouns = [word for (word, pos) in pos_tag(tokenized) if is_noun(pos)]
    return ' '.join(all_nouns)

In [ ]:
# Apply the nouns function to the transcripts to filter only on nouns
speech_df['nouns_adj'] = speech_df.transcript.apply(nouns_adj)
speech_df.head()

,speaker,year,transcript,length,nouns,nouns_adj
0,OPRAH WINFREY,1918.0,thank wallis annenberg special thank dean will...,15301.0,thank wallis dean bay invite today parent facu...,thank wallis special thank dean bay invite tod...
1,OPRAH WINFREY,2007.0,president swygert trustees distinguish guests ...,14168.0,president swygert trustees guests honorees dr ...,president swygert trustees distinguish guests ...
2,FRANKLIN D ROOSEVELT,1932.0,day honorable attainment honor confer upon dee...,16017.0,day attainment honor share satisfaction laurel...,day honorable attainment honor grateful share ...
3,WILLIAM ALLEN,1936.0,commencement orator auditors turn face around ...,14953.0,commencement orator auditors look world think ...,commencement orator auditors look world think ...
4,CARRIE CHAPMAN,1936.0,bring message sweet briar college especially s...,22942.0,message college class portals world message br...,bring message sweet briar college senior class...


In [ ]:
speech_df.to_csv('speech_concise.csv')

In [ ]:
# Create dtm_n (document-term matrix with nouns only)
cv_na = CountVectorizer(stop_words=list(stop_words))
data_cv_na = cv_na.fit_transform(speech_df.nouns_adj)

dtm_na = pd.DataFrame(data_cv_na.toarray(), columns=cv_na.get_feature_names_out())
dtm_na.index = speech_df.index
dtm_na = dtm_na.iloc[:,:-1]
# dtm_na

In [ ]:
# Create the gensim corpus
corpus_na = matutils.Sparse2Corpus(scipy.sparse.csr_matrix(dtm_na.transpose()))

# Create the vocabulary dictionary
id2word_na = dict((v, k) for k, v in cv_na.vocabulary_.items())

In [ ]:
# Let's start with 2 topics
num_topics=2
lda_na = models.LdaModel(corpus=corpus_na, num_topics=num_topics, id2word=id2word_na, passes=10)
get_lda_topics(lda_na, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,school,human,education,college,university,state,country,mean,young,parent
Topic #02,school,thats,job,little,college,didnt,mean,parent,big,write


In [ ]:
# 3 topics
num_topics=3
lda_na = models.LdaModel(corpus=corpus_na, num_topics=num_topics, id2word=id2word_na, passes=10)
get_lda_topics(lda_na, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,school,women,education,college,young,state,university,human,change,country
Topic #02,school,job,thats,little,didnt,dream,parent,college,write,mean
Topic #03,human,college,mean,place,man,little,talk,word,thats,country


In [ ]:
# 4 topics
num_topics=4
lda_na = models.LdaModel(corpus=corpus_na, num_topics=num_topics, id2word=id2word_na, passes=10)
get_lda_topics(lda_na, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,women,school,thats,education,little,country,young,university,state,college
Topic #02,school,job,thats,little,parent,college,thank,didnt,mean,big
Topic #03,college,talk,school,university,job,fear,word,parent,ask,little
Topic #04,human,school,state,college,education,war,mean,women,sense,place


In [ ]:
# 10 topics
num_topics=10
lda_na = models.LdaModel(corpus=corpus_na, num_topics=num_topics, id2word=id2word_na, passes=30)
get_lda_topics(lda_na, num_topics)

,0,1,2,3,4,5,6,7,8,9
Topic #01,cancer,patients,define,women,shine,little,difficult,light,eye,patient
Topic #02,thats,college,dream,women,write,little,generation,state,parent,mean
Topic #03,school,talk,job,word,course,thats,mean,help,little,didnt
Topic #04,parent,job,school,college,thats,mean,little,person,place,moment
Topic #05,job,school,thats,didnt,little,human,best,big,thank,course
Topic #06,women,school,college,country,university,thats,little,young,education,state
Topic #07,human,war,man,peace,state,nations,school,education,college,unite
Topic #08,job,dream,school,parent,important,little,mean,company,college,thats
Topic #09,god,law,jesus,beautiful,harvard,love,school,donõt,advice,child
Topic #10,thats,school,didnt,guy,job,little,thank,young,talk,write


In [ ]:
# 20 topics
num_topics=20
lda_na = models.LdaModel(corpus=corpus_na, num_topics=num_topics, id2word=id2word_na, passes=40)
get_lda_topics(lda_na, num_topics)


,0,1,2,3,4,5,6,7,8,9
Topic #01,college,school,peace,dream,war,job,education,thats,important,little
Topic #02,school,job,college,success,write,place,book,use,education,course
Topic #03,sense,human,civilization,mean,culture,responsibility,television,ask,parent,iõm
Topic #04,parent,human,little,mean,resources,speak,story,age,technology,children
Topic #05,talk,word,thats,school,ask,job,college,mean,write,idea
Topic #06,school,job,little,dream,big,family,change,thats,mean,love
Topic #07,thats,school,college,parent,job,little,mean,didnt,try,course
Topic #08,music,silence,play,musicians,musician,piano,mystery,musical,wonder,successful
Topic #09,school,niagara,cc,banjo,thank,individuals,behalf,vincentian,workshop,houston
Topic #10,person,wrong,man,break,choices,applaud,decision,choice,didnt,dreadful


#NMF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer()
csr_mat = tfidf.fit_transform(speech_df['transcript'])
csr_mat.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
tfidf_dtm = pd.DataFrame(csr_mat.toarray(), columns=tfidf.get_feature_names_out())
tfidf_dtm.index = speech_df.index
tfidf_dtm.iloc[:,:-1]

,aa,aahhhh,aaron,aback,abalthus,abandon,abate,abbreviations,abc,abcsbut,...,ôharvestõ,ôi,ômay,ôsobriety,ôtell,ôthe,ôwe,ôwhat,ôyou,ôyouõre
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.034292,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
318,0.0,0.0,0.027191,0.0,0.0,0.0,0.0,0.0,0.024098,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
319,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
320,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
def tfidf_dtm(df,column_name,add_stop_words=[]):
    """
    Input: corpus (Ex: speech_clean_2, 'transcript')
    Output: Document-Term Matrix (rows: documents, columns: words)

    """
    tfidf = TfidfVectorizer(stop_words=list(stop_words))
    data_tfidf = tfidf.fit_transform(df[column_name])

    tfidf_dtm = pd.DataFrame(data_tfidf.toarray(), columns=tfidf.get_feature_names_out())
    tfidf_dtm.index = df.index

    return tfidf_dtm

In [ ]:
doc_word = tfidf_dtm(speech_df,'transcript')
doc_word.iloc[:,:-1]

,aa,aahhhh,aaron,aback,abalthus,abandon,abate,abbreviations,abc,abcsbut,...,ôharvestõ,ôi,ômay,ôsobriety,ôtell,ôthe,ôwe,ôwhat,ôyou,ôyouõre
0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.037734,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
318,0.0,0.0,0.028648,0.0,0.0,0.0,0.0,0.0,0.025390,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
319,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
320,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Use NMF model, specify number of topics
# (Following Vinny's lecture)
nmf_model = NMF(6, max_iter=800)
doc_topic = nmf_model.fit_transform(doc_word)
doc_topic.shape

(322, 6)

In [ ]:
# Use components in NMF model to find the top 10 words for a given topic
topics = nmf_model.components_.argsort(axis=1)[:,-1:-11:-1]

# Create topic_worrd df
words = doc_word.columns
topic_words = [[words[index] for index in topic] for topic in topics]
pd.DataFrame(topic_words,index=['Topic #' + '{:02d}'.format(i) for i in range(6)])

,0,1,2,3,4,5,6,7,8,9
Topic #00,job,didnt,thats,dream,start,school,ask,thank,parent,kid
Topic #01,government,war,america,nations,country,state,unite,human,peace,education
Topic #02,women,shirtwaist,men,womens,woman,wellesley,factory,triangle,workers,floor
Topic #03,niagara,university,tuskegee,education,vincentian,god,school,thank,catholic,gift
Topic #04,fear,love,music,god,talk,feel,write,try,word,human
Topic #05,genius,scripps,women,lily,necessary,conversation,august,particular,serve,inside


In [ ]:
len(words)

33441